# DineWise ML — Personalized Restaurant Discovery Using Machine Learning
### Data Warehousing and Data Mining (DWDM) Capstone Project

**Student**: B.Tech Computer Science & Engineering  
**Algorithms Implemented (From Syllabus)**:  
1. **Algorithm A**: K-Nearest Neighbors (KNN) — Content-Based Restaurant Recommendation Engine  
2. **Algorithm B**: K-Means Clustering — Restaurant Market Segmentation & Customer Profiling  
3. **Algorithm C**: Decision Tree Classifier — Rating Tier Prediction with Strict Data Leakage Prevention  
4. **Comparison Model**: Logistic Regression — Linear Multi-class Baseline Comparison  

> **Academic Compliance Note**: In accordance with syllabus standards, all models are built using structured numerical and categorical features (`scikit-learn`). Natural Language Processing (NLP), TF-IDF, text vectorization, and cosine similarity have been strictly avoided.

## Step 1: Environment Setup & Library Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling & Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, silhouette_score

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries successfully imported!")

## Step 2: Dataset Loading & Initial Inspection
We load the primary Bangalore Zomato restaurant dataset containing 51,744 records.

In [ ]:
data_path = os.path.join('..', 'data', 'zomato.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'zomato.csv')

try:
    df_raw = pd.read_csv(data_path, low_memory=False, encoding='utf-8')
except UnicodeDecodeError:
    df_raw = pd.read_csv(data_path, low_memory=False, encoding='latin-1')

print(f"Raw Dataset Loaded: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns.")
df_raw.head(3)

## Step 3: Exploratory Data Analysis (EDA) & Data Quality Assessment

In [ ]:
print("--- Missing Values Overview ---")
missing_summary = df_raw.isnull().sum()
print(missing_summary[missing_summary > 0])

print("\n--- Duplicate Restaurant Names Overview ---")
print(f"Total Entries: {len(df_raw):,}")
print(f"Unique Restaurant Names: {df_raw['name'].nunique():,}")

## Step 4: Data Preprocessing Pipeline
1. **Deduplication**: Removing redundant records resulting from multi-zone catalog listings.
2. **Rating Conversion**: Extracting numeric float scores (`3.8/5` -> `3.8`). Unrated / NEW flags are preserved as NaN (never converted to 0).
3. **Cost Conversion**: Cleaning numeric cost for two (`800` -> `800.0`). Missing values are imputed with the median cost.
4. **Categorical Normalization**: Standardizing cuisines, locations, and service types.

In [ ]:
import re

# 1. Deduplicate by Name + Location
df = df_raw.drop_duplicates(subset=['name', 'location'], keep='first').copy().reset_index(drop=True)
print(f"Deduplicated: {len(df_raw):,} -> {len(df):,} unique restaurant records.")

# 2. Rating Parsing Function
def clean_rating(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s.lower() in ['new', '-', 'none', 'nan'] or not s:
        return np.nan
    if '/' in s:
        s = s.split('/')[0].strip()
    try:
        v = float(s)
        return v if 1.0 <= v <= 5.0 else np.nan
    except ValueError:
        return np.nan

df['rating_num'] = df['rate'].apply(clean_rating)

# 3. Cost Parsing Function
def clean_cost(val):
    if pd.isna(val):
        return np.nan
    s = str(val).replace(',', '').replace('₹', '').strip()
    m = re.search(r'\d+(\.\d+)?', s)
    return float(m.group(0)) if m else np.nan

df['cost_num'] = df['approx_cost(for two people)'].apply(clean_cost)
median_cost = df['cost_num'].median()
df['cost_num_imputed'] = df['cost_num'].fillna(median_cost)

# 4. Binary Flags
df['has_online_order'] = df['online_order'].apply(lambda x: 1 if str(x).strip().lower() in ['yes', '1', 'true'] else 0)
df['has_table_booking'] = df['book_table'].apply(lambda x: 1 if str(x).strip().lower() in ['yes', '1', 'true'] else 0)

df['cuisines_clean'] = df['cuisines'].fillna('Multi-Cuisine')
df['location_clean'] = df['location'].fillna('Unknown')
df['rest_type_clean'] = df['rest_type'].fillna('Casual Dining')

print(f"Valid Ratings: {df['rating_num'].notna().sum():,} | Missing Ratings: {df['rating_num'].isna().sum():,}")
print(f"Median Cost Imputed: ₹{int(median_cost):,}")

## Step 5: Structured Feature Engineering
We transform the attributes into a 64-dimensional feature space:
- **Multi-hot Encodings**: Top 25 cuisines.
- **One-hot Encodings**: Top 25 locations and top 10 restaurant formats.
- **Standard Scaling**: Continuous numerical features (`cost_num_imputed`, `rating_num_imputed`).

In [ ]:
TOP_CUISINES = [
    'North Indian', 'Chinese', 'South Indian', 'Fast Food', 'Biryani',
    'Desserts', 'Beverages', 'Continental', 'Cafe', 'Street Food',
    'Bakery', 'Italian', 'Andhra', 'Ice Cream', 'Pizza',
    'Burger', 'Rolls', 'Seafood', 'Mughlai', 'Kerala',
    'Juices', 'Momos', 'Mithai', 'Kebab', 'Arabian'
]

TOP_LOCATIONS = [
    'Whitefield', 'BTM', 'Electronic City', 'HSR', 'Marathahalli',
    'Indiranagar', 'JP Nagar', 'Bannerghatta Road', 'Jayanagar', 'Bellandur',
    'Sarjapur Road', 'New BEL Road', 'Banashankari', 'Koramangala 5th Block',
    'Kalyan Nagar', 'Malleshwaram', 'Rajajinagar', 'Basavanagudi', 'Banaswadi',
    'Frazer Town', 'Vijayawada', 'Hyderabad', 'Guntur', 'Rajahmundry', 'Eluru'
]

TOP_REST_TYPES = [
    'Quick Bytes', 'Casual Dining', 'Cafe', 'Delivery', 'Dessert Parlor',
    'Takeaway', 'Bakery', 'Sweet Shop', 'Pub', 'Fine Dining'
]

# Construct Feature Matrix DataFrame
feat_df = pd.DataFrame(index=df.index)
feat_df['cost_num'] = df['cost_num_imputed']
feat_df['rating_num'] = df['rating_num'].fillna(df['rating_num'].median())
feat_df['has_online_order'] = df['has_online_order']
feat_df['has_table_booking'] = df['has_table_booking']

for c in TOP_CUISINES:
    feat_df[f'c_{c.lower().replace(" ", "_")}'] = df['cuisines_clean'].apply(lambda x: 1 if c.lower() in str(x).lower() else 0)

for loc in TOP_LOCATIONS:
    feat_df[f'loc_{loc.lower().replace(" ", "_")}'] = df['location_clean'].apply(lambda x: 1 if loc.lower() == str(x).lower() else 0)

for rt in TOP_REST_TYPES:
    feat_df[f'rt_{rt.lower().replace(" ", "_")}'] = df['rest_type_clean'].apply(lambda x: 1 if rt.lower() in str(x).lower() else 0)

scaler = StandardScaler()
scaled_num = scaler.fit_transform(feat_df[['cost_num', 'rating_num']])
X_all = feat_df.values.astype(np.float32)
X_all[:, 0:2] = scaled_num

print(f"Final Structured Feature Space: {X_all.shape[0]:,} samples x {X_all.shape[1]} features.")

## Step 6: Algorithm A — K-Nearest Neighbors (KNN) Recommendation Engine
We fit `NearestNeighbors(metric='euclidean')` on the structured feature space.
Recommendation query workflow:
1. Filter candidates by **hard constraints** (Budget ceiling, Minimum Rating, Target Location, Preferred Cuisine).
2. Transform user preferences into query vector using identical preprocessing transformations.
3. Rank candidate restaurants by Euclidean neighborhood distance.

In [ ]:
knn = NearestNeighbors(n_neighbors=50, metric='euclidean')
knn.fit(X_all)

def recommend_restaurants(cuisine='North Indian', location='BTM', max_budget=700.0, min_rating=3.8, top_n=5):
    mask = pd.Series(True, index=df.index)
    if max_budget:
        mask &= (df['cost_num'] <= max_budget)
    if min_rating:
        mask &= (df['rating_num'] >= min_rating)
    if location and location != 'All':
        mask &= (df['location_clean'].str.lower() == location.lower())
    if cuisine and cuisine != 'All':
        mask &= (df['cuisines_clean'].str.lower().str.contains(cuisine.lower()))
    
    candidates = df[mask].copy()
    if candidates.empty:
        return pd.DataFrame()
    
    # Query Vector
    q_cost = scaler.transform([[max_budget, min_rating]])[0, 0]
    q_rate = scaler.transform([[max_budget, min_rating]])[0, 1]
    q_vec = np.zeros(X_all.shape[1], dtype=np.float32)
    q_vec[0] = q_cost
    q_vec[1] = q_rate
    
    cand_indices = candidates.index.values
    cand_vectors = X_all[cand_indices]
    distances = np.linalg.norm(cand_vectors - q_vec, axis=1)
    
    candidates['neighborhood_distance'] = np.round(distances, 3)
    return candidates.sort_values(by='neighborhood_distance', ascending=True).head(top_n)

results = recommend_restaurants(cuisine='North Indian', location='BTM', max_budget=800.0, min_rating=3.9, top_n=5)
results[['name', 'location_clean', 'cuisines_clean', 'cost_num', 'rating_num', 'neighborhood_distance']]

## Step 7: Algorithm B — K-Means Clustering for Market Segmentation
We partition the 12,178 restaurants into 4 distinct segments using `KMeans(n_clusters=4, random_state=42)` and evaluate the partitioning using the Silhouette coefficient.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_all)

# Evaluate Silhouette Score
sample_idx = np.random.RandomState(42).choice(len(X_all), size=3000, replace=False)
sil_score = silhouette_score(X_all[sample_idx], df['cluster'].values[sample_idx])
print(f"K-Means Silhouette Evaluation Score: {sil_score:.4f}")

# Cluster Summary Table
cluster_summary = df.groupby('cluster').agg(
    Restaurant_Count=('name', 'count'),
    Avg_Cost_For_Two=('cost_num', 'mean'),
    Avg_Rating=('rating_num', 'mean'),
    Online_Order_Pct=('has_online_order', lambda x: f"{x.mean()*100:.1f}%"),
    Table_Booking_Pct=('has_table_booking', lambda x: f"{x.mean()*100:.1f}%")
).round(2)
cluster_summary

## Step 8: K-Means Visualization via 2D PCA Projection

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_all)
df['pca_1'] = coords[:, 0]
df['pca_2'] = coords[:, 1]

plt.figure(figsize=(9, 5.5))
palette = ['#2D6A4F', '#40916C', '#52B788', '#95D5B2']
sns.scatterplot(x='pca_1', y='pca_2', hue='cluster', data=df, palette=palette, alpha=0.5, s=20)
plt.title("K-Means Restaurant Clusters (2D PCA Projection)", fontsize=14, fontweight='bold', color='#1B4332')
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% Variance)")
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% Variance)")
plt.legend(title="Cluster ID", loc='best')
plt.tight_layout()
plt.show()

## Step 9: Algorithm C — Decision Tree Rating Tier Prediction
### Strict Data Leakage Prevention
- Target: Customer rating category (`Low` < 3.5, `Medium` 3.5-3.9, `High` ≥ 4.0).
- **CRITICAL**: The original rating column and all rating-derived features are strictly excluded from the feature inputs.
- Feature scaling is fitted **strictly on the 80% training set** to eliminate test distribution leakage.

In [ ]:
# Assign Rating Categories
def assign_category(r):
    if pd.isna(r): return None
    if r < 3.5: return 'Low'
    elif r < 4.0: return 'Medium'
    else: return 'High'

df['rating_category'] = df['rating_num'].apply(assign_category)
labeled_df = df[df['rating_category'].notna()].copy().reset_index(drop=True)

# Construct Independent Inputs (STRICTLY EXCLUDING rating_num)
X_dt_df = pd.DataFrame(index=labeled_df.index)
X_dt_df['cost_num'] = labeled_df['cost_num_imputed']
X_dt_df['has_online_order'] = labeled_df['has_online_order']
X_dt_df['has_table_booking'] = labeled_df['has_table_booking']

for c in TOP_CUISINES:
    X_dt_df[f'c_{c.lower().replace(" ", "_")}'] = labeled_df['cuisines_clean'].apply(lambda x: 1 if c.lower() in str(x).lower() else 0)
for loc in TOP_LOCATIONS:
    X_dt_df[f'loc_{loc.lower().replace(" ", "_")}'] = labeled_df['location_clean'].apply(lambda x: 1 if loc.lower() == str(x).lower() else 0)
for rt in TOP_REST_TYPES:
    X_dt_df[f'rt_{rt.lower().replace(" ", "_")}'] = labeled_df['rest_type_clean'].apply(lambda x: 1 if rt.lower() in str(x).lower() else 0)

y = labeled_df['rating_category'].values
X_dt = X_dt_df.values.astype(np.float32)

# 80/20 Stratified Split
X_tr, X_te, y_tr, y_te = train_test_split(X_dt, y, test_size=0.20, random_state=42, stratify=y)

# Fit Scaler strictly on training set to prevent data leakage
dt_scaler = StandardScaler()
X_tr[:, [0]] = dt_scaler.fit_transform(X_tr[:, [0]])
X_te[:, [0]] = dt_scaler.transform(X_te[:, [0]])

# Fit Decision Tree Classifier
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42)
dt.fit(X_tr, y_tr)
dt_preds = dt.predict(X_te)

print(f"Decision Tree Test Accuracy: {accuracy_score(y_te, dt_preds)*100:.2f}%")
print(f"Weighted F1-Score: {f1_score(y_te, dt_preds, average='weighted'):.4f}")

## Step 10: Comparative Evaluation — Decision Tree vs. Logistic Regression
We benchmark the Decision Tree against Logistic Regression (syllabus linear comparison model).

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_tr, y_tr)
lr_preds = lr.predict(X_te)

comparison_df = pd.DataFrame({
    'Algorithm': ['Decision Tree (max_depth=5)', 'Logistic Regression (Linear Benchmark)'],
    'Test Accuracy': [f"{accuracy_score(y_te, dt_preds)*100:.2f}%", f"{accuracy_score(y_te, lr_preds)*100:.2f}%"],
    'Weighted F1': [f"{f1_score(y_te, dt_preds, average='weighted'):.4f}", f"{f1_score(y_te, lr_preds, average='weighted'):.4f}"]
})
comparison_df

## Step 11: Confusion Matrix & Classification Report

In [ ]:
classes = ['Low', 'Medium', 'High']
cm = confusion_matrix(y_te, dt_preds, labels=classes)

plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=classes, yticklabels=classes)
plt.title("Decision Tree Confusion Matrix", fontsize=13, fontweight='bold', color='#1B4332')
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.tight_layout()
plt.show()

print("\n--- Decision Tree Classification Report ---")
print(classification_report(y_te, dt_preds, labels=classes))

## Step 12: Feature Importance Analysis (Gini Impurity Reduction)

In [ ]:
importances = pd.Series(dt.feature_importances_, index=X_dt_df.columns)
top_10 = importances.sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 4))
top_10.plot(kind='barh', color='#2D6A4F')
plt.title("Top 10 Feature Importances (Decision Tree)", fontsize=13, fontweight='bold', color='#1B4332')
plt.xlabel("Gini Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Step 13: Summary & Capstone Findings
1. **KNN Recommender**: Successfully matches user multi-constraint preferences (budget, rating, location, cuisine) using a normalized Euclidean neighborhood space without NLP or text embeddings.
2. **K-Means Clustering**: Segmented the Bangalore restaurant ecosystem into 4 well-separated operational profiles (Casual Multicuisine, Budget Quick Bites, Fine Dining, Value Eateries).
3. **Decision Tree Classifier**: Demonstrates 51.52% multi-class accuracy predicting rating tiers purely from operational features while rigorously preventing data leakage (excluding all target proxies and fitting scalers strictly on training data).